# Three unconditional generation baselines

Select a GPU runtime and run the first three code cells. This notebook trains PaCoDi SDE, FreqDiff, and TimeBridge GP in sequence. It uses the existing ETTh1 data split, 128-point windows, seed 42, and the saved run's optimizer-update budget. The models use their own objectives and samplers.

The report compares the three baselines with the completed Diffusion-TS and overlap models using the same held-out test windows and evaluation settings.


In [ ]:
from google.colab import drive, userdata
from pathlib import Path
import getpass
import io
import json
import os
import subprocess
import sys
import urllib.error
import urllib.request
import zipfile

drive.mount('/content/drive')
REPOSITORY = 'Zhou-Leheng/adaptive-time-frequency-generation'
request = urllib.request.Request(
    f'https://api.github.com/repos/{REPOSITORY}/zipball/main',
    headers={'Accept': 'application/vnd.github+json'},
)
try:
    response = urllib.request.urlopen(request, timeout=90)
except urllib.error.HTTPError as error:
    if error.code not in (401, 404):
        raise
    try:
        github_token = userdata.get('GITHUB_TOKEN')
    except Exception:
        github_token = getpass.getpass('GitHub token with repository contents read access: ')
    request.add_header('Authorization', f'Bearer {github_token}')
    response = urllib.request.urlopen(request, timeout=90)
    del github_token

with response:
    archive = zipfile.ZipFile(io.BytesIO(response.read()))
del request, response
root = Path('/content/atfg_code')
root.mkdir(exist_ok=True)
for member in archive.infolist():
    target = (root / member.filename).resolve()
    if not target.is_relative_to(root.resolve()):
        raise ValueError('Unexpected archive path')
archive.extractall(root)
CODE = root / archive.namelist()[0].split('/')[0]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(CODE / 'requirements-external.txt')])
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime before starting the experiment.'
print('GPU:', torch.cuda.get_device_name(0))
print('Code:', CODE)

In [ ]:
RUN_NAME = 'etth1_seed42'
RUN_METHODS = ['pacodi_sde', 'freqdiff', 'timebridge_gp']
TRAIN_STEPS = None  # None inherits 5000 updates from the saved pilot.
RESULT_NAME = 'benchmark_three_baselines'
RESULTS = Path('/content/drive/MyDrive/ATFG') / RUN_NAME
assert (RESULTS / 'settings.json').is_file(), 'Select the completed ETTh1 experiment.'
assert (RESULTS / 'benchmark_diffusion_ts' / 'shared_ts2vec.pt').is_file(), 'Complete the original comparison.'
assert (RESULTS / 'overlap_output' / 'evaluation' / 'patch_overlap.json').is_file(), 'Complete the patch overlap evaluation.'
settings = json.loads((RESULTS / 'settings.json').read_text())
OUTPUT = RESULTS / RESULT_NAME
LOG = Path('/content') / f'atfg_{RESULT_NAME}_{RUN_NAME}.log'
PID_RECORD = Path('/content') / f'atfg_{RESULT_NAME}_{RUN_NAME}_pid.json'
print('Models:', ', '.join(RUN_METHODS))
print('Optimizer updates per model:', TRAIN_STEPS or settings['steps'])
print('Output:', OUTPUT)


In [ ]:
from filelock import FileLock

with FileLock(str(RESULTS / '.run.lock'), timeout=0):
    pass
runner = CODE / 'run_external.py'
command = [
    'nohup', sys.executable, '-u', str(runner),
    '--results', str(RESULTS), '--name', RESULT_NAME,
    '--data', '/content/ETTh1.csv', '--download', '--device', 'cuda',
    '--methods', *RUN_METHODS,
]
if TRAIN_STEPS is not None:
    command.extend(['--train-steps', str(TRAIN_STEPS)])
with LOG.open('a') as handle:
    process = subprocess.Popen(command, cwd=CODE, stdout=handle, stderr=subprocess.STDOUT,
                               stdin=subprocess.DEVNULL, start_new_session=True)
PID_RECORD.write_text(json.dumps({'pid': process.pid, 'script': str(runner)}))
print('Comparison PID:', process.pid)


In [ ]:
if LOG.exists():
    print(LOG.read_text(errors='replace')[-12000:])
else:
    print('Launch the preceding cell first.')


In [ ]:
import pandas as pd
from IPython.display import display

table = OUTPUT / 'comparison.csv'
if table.is_file():
    results = pd.read_csv(table)
    display(results[['method', 'context_fid', 'predictive_mean', 'correlational_score',
                     'discriminative_mean', 'swd', 'psd_relative_l1', 'patch_boundary_ratio']].round(4))
    pending = set(RUN_METHODS) - set(results['method'])
    print('Pending:', ', '.join(sorted(pending)) if pending else 'none')
    print('Full results:', table)
else:
    print('The first new model is training or awaiting evaluation.')


Run the same launch cell to resume saved training and evaluation. To choose a different training budget, use a different RESULT_NAME. To stop and save at the next training boundary, set STOP = True below.


In [ ]:
STOP = False
if STOP:
    import signal
    record = json.loads(PID_RECORD.read_text())
    command_line = Path(f"/proc/{record['pid']}/cmdline")
    if command_line.exists() and record['script'].encode() in command_line.read_bytes().split(b'\x00'):
        os.kill(record['pid'], signal.SIGTERM)
        print('Stop requested. Read the log for the checkpoint confirmation.')
    else:
        print('The recorded process has already exited.')
